# 10 — Hyperbolic quadrant Step 2: dual-channel RBM training

Train two **independent** real-valued Bernoulli RBMs on Step 1 softmax tensors:

- **RBM1** on channel 1 (`e+`)
- **RBM2** on channel 2 (`e-`)

Each visible sample is a **flattened** `(2000 × 10)` one-hot block → **20,000** visible units.  
We store the **cumulative** mini-batch \(\Delta W\) at the end of each epoch (before it is added to \(W\)).

In [14]:
from pathlib import Path

import numpy as np

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

proc = root / "data" / "processed"
path_c1 = proc / "channel1_softmax.npy"
path_c2 = proc / "channel2_softmax.npy"

for p in (path_c1, path_c2):
    assert p.exists(), f"Missing {p} — run notebook 09 (Step 1) first."

N_VISIBLE = 20_000
N_HIDDEN = 128
LR = 0.01
BATCH_SIZE = 10
EPOCHS = 50
INIT_SEED = 42


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def flatten_channel(arr):
    """(n_users, n_movies, K) -> (n_users, n_movies * K)."""
    n_users, n_movies, k = arr.shape
    assert k * n_movies == N_VISIBLE, f"Expected {N_VISIBLE} visible units, got {k * n_movies}"
    return arr.reshape(n_users, N_VISIBLE).astype(np.float64)


channel1 = np.load(path_c1)
channel2 = np.load(path_c2)
X1 = flatten_channel(channel1)
X2 = flatten_channel(channel2)

print(f"channel1 tensor: {channel1.shape} -> X1 {X1.shape}")
print(f"channel2 tensor: {channel2.shape} -> X2 {X2.shape}")
print(f"Non-zero visibles per user (ch1): {(X1.sum(axis=1) > 0).sum()} users with ratings")

channel1 tensor: (51, 2000, 10) -> X1 (51, 20000)
channel2 tensor: (51, 2000, 10) -> X2 (51, 20000)
Non-zero visibles per user (ch1): 51 users with ratings


## CD-1 training with per-epoch \(\Delta W\) logging

In [11]:
def reconstruction_mse(X, W, b_h):
    """MSE between data and mean-field reconstruction P(v|h(data))."""
    h_prob = sigmoid(X @ W + b_h)
    v_prob = sigmoid(h_prob @ W.T)
    return float(np.mean((X - v_prob) ** 2))


def train_rbm(X, seed, channel_name):
    """Train one RBM; return W, b_h, deltaW_per_epoch, mse_log dict."""
    rng = np.random.default_rng(seed)
    n_samples, n_visible = X.shape
    n_hidden = N_HIDDEN

    W = rng.normal(0.0, 0.01, size=(n_visible, n_hidden))
    b_h = np.zeros(n_hidden, dtype=np.float64)

    deltaW_per_epoch = np.zeros((EPOCHS, n_visible, n_hidden), dtype=np.float32)
    mse_log = {}

    print(f"\n=== Training {channel_name} ===")
    print(f"  samples={n_samples}, visible={n_visible}, hidden={n_hidden}")
    print(f"  lr={LR}, batch={BATCH_SIZE}, epochs={EPOCHS}, init_seed={seed}")

    for epoch in range(EPOCHS):
        epoch_delta = np.zeros_like(W)
        order = rng.permutation(n_samples)

        for start in range(0, n_samples, BATCH_SIZE):
            idx = order[start : start + BATCH_SIZE]
            v_data = X[idx]  # (batch, V)
            bs = v_data.shape[0]

            # CD-1 positive phase
            h_prob = sigmoid(v_data @ W + b_h)
            h_data = (rng.random(h_prob.shape) < h_prob).astype(np.float64)

            # CD-1 negative phase (one Gibbs step)
            v_recon_prob = sigmoid(h_data @ W.T)
            v_recon = (rng.random(v_recon_prob.shape) < v_recon_prob).astype(np.float64)
            h_recon_prob = sigmoid(v_recon @ W + b_h)
            h_recon = (rng.random(h_recon_prob.shape) < h_recon_prob).astype(np.float64)

            # delta_W = lr * (v_data.T @ h_data - v_recon.T @ h_recon) / batch_size
            pos = v_data.T @ h_data
            neg = v_recon.T @ h_recon
            dW = LR * (pos - neg) / bs
            epoch_delta += dW

            db_h = LR * (h_prob.mean(axis=0) - h_recon_prob.mean(axis=0))
            b_h += db_h

        deltaW_per_epoch[epoch] = epoch_delta.astype(np.float32)
        W += epoch_delta

        ep = epoch + 1
        if ep in (1, 25, EPOCHS):
            mse = reconstruction_mse(X, W, b_h)
            mse_log[ep] = mse
            print(f"  Epoch {ep:02d} | reconstruction MSE: {mse:.6f}")

    return W, b_h, deltaW_per_epoch, mse_log


W1, bh1, deltaW1, mse1 = train_rbm(X1, seed=INIT_SEED, channel_name="RBM1 (e+)")
W2, bh2, deltaW2, mse2 = train_rbm(X2, seed=INIT_SEED, channel_name="RBM2 (e-)")


=== Training RBM1 (e+) ===
  samples=51, visible=20000, hidden=128
  lr=0.01, batch=10, epochs=50, init_seed=42
  Epoch 01 | reconstruction MSE: 0.132426
  Epoch 25 | reconstruction MSE: 0.054427
  Epoch 50 | reconstruction MSE: 0.022274

=== Training RBM2 (e-) ===
  samples=51, visible=20000, hidden=128
  lr=0.01, batch=10, epochs=50, init_seed=42
  Epoch 01 | reconstruction MSE: 0.131599
  Epoch 25 | reconstruction MSE: 0.055136
  Epoch 50 | reconstruction MSE: 0.022300


## Save artifacts

In [12]:
paths = {
    "deltaW1": proc / "deltaW1_per_epoch.npy",
    "deltaW2": proc / "deltaW2_per_epoch.npy",
    "W1": proc / "rbm1_weights_final.npy",
    "W2": proc / "rbm2_weights_final.npy",
    "bh1": proc / "rbm1_bias_hidden_final.npy",
    "bh2": proc / "rbm2_bias_hidden_final.npy",
}

np.save(paths["deltaW1"], deltaW1)
np.save(paths["deltaW2"], deltaW2)
np.save(paths["W1"], W1)
np.save(paths["W2"], W2)
np.save(paths["bh1"], bh1)
np.save(paths["bh2"], bh2)

for k, p in paths.items():
    print(f"Saved {k}: {p}")

Saved deltaW1: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/deltaW1_per_epoch.npy
Saved deltaW2: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/deltaW2_per_epoch.npy
Saved W1: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbm1_weights_final.npy
Saved W2: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbm2_weights_final.npy
Saved bh1: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbm1_bias_hidden_final.npy
Saved bh2: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/rbm2_bias_hidden_final.npy


## Verification

In [13]:
def weight_stats(W, name):
    print(f"\n{name} final W: shape={W.shape}")
    print(f"  mean={W.mean():.6f}  std={W.std():.6f}  min={W.min():.6f}  max={W.max():.6f}")


for tag, dW, mse_log, W, bh in [
    ("RBM1 (e+)", deltaW1, mse1, W1, bh1),
    ("RBM2 (e-)", deltaW2, mse2, W2, bh2),
]:
    print(f"\n{'=' * 50}")
    print(tag)
    print(f"delta_W per epoch shape: {dW.shape}  (expected ({EPOCHS}, {N_VISIBLE}, {N_HIDDEN}))")
    assert dW.shape == (EPOCHS, N_VISIBLE, N_HIDDEN)
    print("Reconstruction MSE:")
    for ep in (1, 25, EPOCHS):
        print(f"  epoch {ep:2d}: {mse_log[ep]:.6f}")
    weight_stats(W, tag)
    print(f"b_h shape: {bh.shape}, mean={bh.mean():.6f}")

print("\nDone.")


RBM1 (e+)
delta_W per epoch shape: (50, 20000, 128)  (expected (50, 20000, 128))
Reconstruction MSE:
  epoch  1: 0.132426
  epoch 25: 0.054427
  epoch 50: 0.022274

RBM1 (e+) final W: shape=(20000, 128)
  mean=-0.021352  std=0.018560  min=-0.084191  max=0.556552
b_h shape: (128,), mean=2.032228

RBM2 (e-)
delta_W per epoch shape: (50, 20000, 128)  (expected (50, 20000, 128))
Reconstruction MSE:
  epoch  1: 0.131599
  epoch 25: 0.055136
  epoch 50: 0.022300

RBM2 (e-) final W: shape=(20000, 128)
  mean=-0.021348  std=0.018626  min=-0.083033  max=0.571098
b_h shape: (128,), mean=2.042137

Done.
